# GBasis Tutorial - Two-Electron Integral Optimizations

This tutorial covers the **optimized two-electron integral computation** in GBasis, including:

1. **OS+HGP Algorithm** - Obara-Saika + Head-Gordon-Pople recursion scheme
2. **Boys Functions** - Standard Coulomb and range-separated (erf/erfc) potentials
3. **Schwarz Screening** - Skip negligible integrals for speedup
4. **High Angular Momentum** - Stable computation for d/f orbitals
5. **PySCF Comparison** - Validation against a trusted reference

These optimizations are implemented as part of **Issue #221** to improve 2-electron integral performance.

## 1. Setup and Imports

In [ ]:
import numpy as np
import time

# GBasis imports
from gbasis.contractions import GeneralizedContractionShell
from gbasis.integrals.electron_repulsion import (
    ElectronRepulsionIntegral,
    ElectronRepulsionIntegralImproved,
    electron_repulsion_integral,
    electron_repulsion_integral_improved,
)

# Low-level OS+HGP algorithm
from gbasis.integrals._two_elec_int_improved import compute_two_electron_integrals_os_hgp
from gbasis.integrals.point_charge import PointChargeIntegral

# Schwarz screening
from gbasis.integrals._screening import (
    SchwarzScreener,
    shell_pair_significant,
    compute_schwarz_bounds,
)

# Boys functions for different potentials
from gbasis.integrals.boys_functions import (
    boys_function_standard,
    boys_function_erf,
    boys_function_erfc,
    get_boys_function,
)

print("All imports successful!")

## 2. Creating Test Molecules

We'll create hydrogen chain molecules to demonstrate the optimizations.
Larger separations between atoms lead to more sparsity and better screening.

In [ ]:
def make_shell(coord, angmom, exps, coeffs):
    """Create a GeneralizedContractionShell."""
    return GeneralizedContractionShell(
        angmom,
        np.array(coord),
        np.array(coeffs).reshape(-1, 1),
        np.array(exps),
        'cartesian'
    )

def make_h_chain(n_atoms, spacing):
    """Create a hydrogen chain with STO-3G basis."""
    # STO-3G exponents and coefficients for H
    exps = [3.42525091, 0.62391373, 0.16885540]
    coeffs = [0.15432897, 0.53532814, 0.44463454]
    
    basis = []
    for i in range(n_atoms):
        coord = [i * spacing, 0.0, 0.0]
        basis.append(make_shell(coord, 0, exps, coeffs))
    
    return basis

# Create test molecules
h4_compact = make_h_chain(4, spacing=1.4)   # Compact: H-H = 1.4 Bohr
h6_extended = make_h_chain(6, spacing=15.0) # Extended: H-H = 15 Bohr

print(f"Compact H4:  {len(h4_compact)} shells, 1.4 Bohr spacing")
print(f"Extended H6: {len(h6_extended)} shells, 15.0 Bohr spacing")

## 3. The OS+HGP Algorithm

The improved implementation uses the **Obara-Saika + Head-Gordon-Pople** algorithm:

```
Step 1: Boys Function       F_m(T) = initial values
Step 2: VRR (Vertical)      [a0|00]^m from [00|00]^m
Step 3: ETR (Electron)      [a0|c0]^0 from [a0|00]^m  
Step 4: Contract            Sum over primitives (HGP: do early!)
Step 5: HRR (Horizontal)    [ab|cd] from [a0|c0] (HGP: do last!)
```

### Key Optimization (HGP):
The **Head-Gordon-Pople** optimization does HRR **after** contraction.
This is more efficient because contracted integrals are smaller than primitive integrals.

In [ ]:
print("""
Obara-Saika + Head-Gordon-Pople Algorithm
==========================================

Traditional:           HGP Optimized:
------------           --------------
Boys F_m(T)            Boys F_m(T)
    |                      |
   VRR                    VRR
    |                      |
   ETR                    ETR
    |                      |
   HRR  <-- expensive!    Contract  <-- do this first!
    |                      |
 Contract                 HRR  <-- now cheaper!
    |                      |
 [ab|cd]                [ab|cd]

Why is HGP better?
- HRR has NO 'm' index (no auxiliary function dependence)
- After contraction, fewer terms to process
- Especially beneficial for contracted basis sets (like cc-pVTZ)
""")

### Comparing Original vs Improved Implementation

In [ ]:
# Compare timing: original vs improved
basis = make_h_chain(4, spacing=1.4)

# Original implementation
start = time.perf_counter()
eri_old = electron_repulsion_integral(basis, notation="chemist")
time_old = time.perf_counter() - start

# Improved OS+HGP implementation
start = time.perf_counter()
eri_new = electron_repulsion_integral_improved(basis, notation="chemist")
time_new = time.perf_counter() - start

# Verify they match
max_diff = np.max(np.abs(eri_old - eri_new))

print(f"Original implementation: {time_old:.4f} seconds")
print(f"Improved (OS+HGP):      {time_new:.4f} seconds")
print(f"Max difference:         {max_diff:.2e}")
print(f"Results match:          {np.allclose(eri_old, eri_new, rtol=1e-10)}")

## 4. Boys Functions for Different Potentials

GBasis supports various two-electron potentials beyond the standard Coulomb $1/r_{12}$:

| Potential | Formula | Use Case |
|-----------|---------|----------|
| **Coulomb** | $1/r_{12}$ | Standard electron repulsion |
| **erf (long-range)** | $\text{erf}(\omega r)/r$ | Range-separated hybrids (LC-) |
| **erfc (short-range)** | $\text{erfc}(\omega r)/r$ | Short-range exchange |

The key identity: $\text{erf}(\omega r)/r + \text{erfc}(\omega r)/r = 1/r$

In [ ]:
# Compare Boys functions for different potentials
T_values = np.array([0.0, 0.5, 1.0, 2.0, 5.0])
m = np.array([0])  # F_0(T)
omega = 0.5  # Range-separation parameter

print("Boys Function F_0(T) for Different Potentials:")
print("-" * 50)
print(f"{'T':>8} {'Coulomb':>12} {'erf':>12} {'erfc':>12}")
print("-" * 50)

for t in T_values:
    t_arr = np.array([t])
    rho = np.array([0.8])
    
    coulomb = boys_function_standard(m, t_arr)[0]
    erf_val = boys_function_erf(m, t_arr, rho, omega)[0]
    erfc_val = boys_function_erfc(m, t_arr, rho, omega)[0]
    
    print(f"{t:>8.1f} {coulomb:>12.6f} {erf_val:>12.6f} {erfc_val:>12.6f}")

print("\nNote: erf + erfc = Coulomb (range separation identity)")

In [ ]:
# Verify: erf + erfc = Coulomb
T_test = np.array([1.0])
rho = np.array([0.8])

coulomb = boys_function_standard(m, T_test)[0]
erf_val = boys_function_erf(m, T_test, rho, omega)[0]
erfc_val = boys_function_erfc(m, T_test, rho, omega)[0]

print("Range Separation Verification:")
print(f"  Coulomb:     {coulomb:.10f}")
print(f"  erf + erfc:  {erf_val + erfc_val:.10f}")
print(f"  Difference:  {abs(coulomb - (erf_val + erfc_val)):.2e}")

### Using the `get_boys_function` Factory

In [ ]:
# Factory function for different potentials
print("Using get_boys_function() factory:")
print("-" * 40)

# Standard Coulomb
boys_coulomb = get_boys_function("coulomb")
print(f"Coulomb: F_0(1.0) = {boys_coulomb(np.array([0]), np.array([1.0]))[0]:.6f}")

# Long-range (erf)
boys_lr = get_boys_function("erf", omega=0.5)
print(f"Long-range (omega=0.5): F_0(1.0) = {boys_lr(np.array([0]), np.array([1.0]), rho=np.array([0.8]))[0]:.6f}")

# Short-range (erfc)
boys_sr = get_boys_function("erfc", omega=0.5)
print(f"Short-range (omega=0.5): F_0(1.0) = {boys_sr(np.array([0]), np.array([1.0]), rho=np.array([0.8]))[0]:.6f}")

print("\nAvailable potentials: 'coulomb', 'erf', 'erfc'")

## 5. Schwarz Screening

### What is Schwarz Screening?

The **Schwarz inequality** states:

$$|(ab|cd)| \leq \sqrt{(ab|ab)} \times \sqrt{(cd|cd)}$$

If this upper bound is smaller than a threshold (e.g., $10^{-12}$), we can skip the integral computation entirely!

### Benefits:
- For extended molecules, 60-90% of integrals can be skipped
- Speedup of 2-10x depending on system size and geometry

In [ ]:
# Demonstrate Schwarz screening on extended system
boys_func = PointChargeIntegral.boys_func

print("=" * 55)
print("Schwarz Screening Comparison")
print("=" * 55)

for name, basis, spacing in [
    ("Compact H4 (1.4 Bohr)", h4_compact, 1.4),
    ("Extended H6 (15.0 Bohr)", h6_extended, 15.0),
]:
    screener = SchwarzScreener(
        basis, boys_func, compute_two_electron_integrals_os_hgp
    )
    
    n = len(basis)
    for i in range(n):
        for j in range(n):
            for k in range(n):
                for l in range(n):
                    screener.is_significant(i, j, k, l)
    
    stats = screener.get_statistics()
    
    print(f"\n{name}:")
    print(f"  Total shell quartets:  {stats['total']}")
    print(f"  Computed:              {stats['n_computed']}")
    print(f"  Screened (skipped):    {stats['n_screened']}")
    print(f"  Percent screened:      {stats['percent_screened']:.1f}%")
    print(f"  Speedup factor:        {stats['speedup_factor']:.2f}x")

### Shell-Pair Prescreening

Before even computing Schwarz bounds, we can use the **Gaussian product theorem** to
quickly check if a shell pair is significant:

$$\exp\left(-\frac{\alpha_a \alpha_b}{\alpha_a + \alpha_b} |\mathbf{A} - \mathbf{B}|^2\right)$$

If this decay factor is below threshold for all primitive pairs, the shell pair contributes nothing.

In [ ]:
# Demonstrate shell-pair prescreening
print("Shell-Pair Prescreening:")
print("-" * 50)

# Same center - always significant
shell_a = make_shell([0, 0, 0], 0, [1.0], [1.0])
shell_b = make_shell([0, 0, 0], 0, [0.5], [1.0])
print(f"Same center (0,0,0):         {shell_pair_significant(shell_a, shell_b)}")

# Close shells
shell_c = make_shell([1.0, 0, 0], 0, [1.0], [1.0])
print(f"Close (1.0 Bohr apart):      {shell_pair_significant(shell_a, shell_c)}")

# Far apart
shell_d = make_shell([50.0, 0, 0], 0, [1.0], [1.0])
print(f"Far apart (50 Bohr):         {shell_pair_significant(shell_a, shell_d)}")

# Very diffuse (small exponent) at distance
shell_e = make_shell([10.0, 0, 0], 0, [0.01], [1.0])
print(f"Diffuse at 10 Bohr (a=0.01): {shell_pair_significant(shell_a, shell_e)}")

## 6. High Angular Momentum Integrals

The improved algorithm correctly handles **d-orbitals (L=2)** and **f-orbitals (L=3)** without numerical overflow.

In [ ]:
coord = np.array([0.0, 0.0, 0.0])
exps = np.array([0.5])
coeffs = np.array([[1.0]])

# Angular momentum components
orbitals = {
    's (L=0)': (0, np.array([[0, 0, 0]])),
    'p (L=1)': (1, np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]])),
    'd (L=2)': (2, np.array([[2, 0, 0], [1, 1, 0], [1, 0, 1], [0, 2, 0], [0, 1, 1], [0, 0, 2]])),
    'f (L=3)': (3, np.array([[3, 0, 0], [2, 1, 0], [2, 0, 1], [1, 2, 0], [1, 1, 1],
                            [1, 0, 2], [0, 3, 0], [0, 2, 1], [0, 1, 2], [0, 0, 3]]))
}

print("High Angular Momentum Test:")
print("-" * 60)
print(f"{'Orbital':>10} {'Shape':>20} {'Has NaN':>10} {'Has Inf':>10}")
print("-" * 60)

for name, (angmom, comp) in orbitals.items():
    result = compute_two_electron_integrals_os_hgp(
        PointChargeIntegral.boys_func,
        coord, angmom, comp, exps, coeffs,
        coord, angmom, comp, exps, coeffs,
        coord, angmom, comp, exps, coeffs,
        coord, angmom, comp, exps, coeffs,
    )
    
    shape_str = str(result.shape[:4])
    has_nan = np.any(np.isnan(result))
    has_inf = np.any(np.isinf(result))
    
    print(f"{name:>10} {shape_str:>20} {str(has_nan):>10} {str(has_inf):>10}")

print("\nAll integrals computed successfully without numerical issues!")

## 7. Validation Against PySCF

We validate our implementation against PySCF, a trusted quantum chemistry package.
Both the original and improved implementations should match PySCF exactly.

In [ ]:
try:
    from pyscf import gto
    HAS_PYSCF = True
except ImportError:
    HAS_PYSCF = False
    print("PySCF not installed. Install with: pip install pyscf")

if HAS_PYSCF:
    # H2 with STO-3G basis
    exps_h = np.array([3.42525091, 0.62391373, 0.16885540])
    coeffs_h = np.array([[0.15432897], [0.53532814], [0.44463454]])
    
    basis_gbasis = [
        GeneralizedContractionShell(0, np.array([0.0, 0.0, 0.0]), coeffs_h, exps_h, 'cartesian'),
        GeneralizedContractionShell(0, np.array([1.4, 0.0, 0.0]), coeffs_h, exps_h, 'cartesian'),
    ]
    
    mol = gto.M(
        atom='H 0 0 0; H 1.4 0 0', unit='bohr',
        basis={'H': gto.basis.parse(
            'H  S\n    3.42525091    0.15432897\n    0.62391373    0.53532814\n    0.16885540    0.44463454'
        )}, verbose=0,
    )
    
    # PySCF returns chemist notation (ij|kl)
    eri_pyscf = mol.intor('int2e')
    
    # GBasis - both implementations in chemist notation
    eri_original = electron_repulsion_integral(basis_gbasis, notation='chemist')
    eri_improved = electron_repulsion_integral_improved(basis_gbasis, notation='chemist')
    
    print("H2/STO-3G Comparison (Chemist notation):")
    print("=" * 55)
    print(f"{'Integral':>12} {'PySCF':>12} {'Original':>12} {'Improved':>12}")
    print("-" * 55)
    
    for i in range(2):
        for j in range(2):
            label = f"({i}{i}|{j}{j})"
            print(f"{label:>12} {eri_pyscf[i,i,j,j]:>12.6f} {eri_original[i,i,j,j]:>12.6f} {eri_improved[i,i,j,j]:>12.6f}")
    
    print(f"\nMax |Original - PySCF|:    {np.max(np.abs(eri_original - eri_pyscf)):.2e}")
    print(f"Max |Improved - PySCF|:    {np.max(np.abs(eri_improved - eri_pyscf)):.2e}")
    print(f"Max |Improved - Original|: {np.max(np.abs(eri_improved - eri_original)):.2e}")

## 8. Summary

### Key Features Implemented (Issue #221):

| Week | Feature | Benefit |
|------|---------|--------|
| 1 | Boys Function | Efficient $F_m(T)$ via hyp1f1 for all orders |
| 2 | VRR (Vertical Recursion) | Build angular momentum on center A |
| 3 | ETR + Contraction | Transfer to center C + sum over primitives |
| 4 | HRR (Horizontal Recursion) | Distribute to B and D (post-contraction per HGP) |
| 5 | erf/erfc Boys Functions | Range-separated DFT support |
| 6 | GBasis Integration | `ElectronRepulsionIntegralImproved` class |
| 7 | Schwarz Screening | Skip negligible integrals (2-10x speedup) |
| 8 | PySCF Validation | Verified against trusted reference |

### When to Use Screening:
- **Large molecules** with spatially separated atoms
- **Extended basis sets** (cc-pVDZ, cc-pVTZ, etc.)
- **Periodic systems** or **clusters**

### When Screening Has Less Effect:
- **Compact molecules** where all atoms are close
- **Minimal basis sets** (STO-3G) with few primitives

In [ ]:
print("""
Thanks for completing this tutorial!

For more information:
- Documentation: http://gbasis.qcdevs.org/
- GitHub: https://github.com/theochem/gbasis
- Issue #221: Two-electron integral improvements
""")